## CLASSIFICATION

In [13]:
# Load Data, Separate Features/Target, and Stratified Split
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Load cleaned dataset
df = pd.read_csv("../data/processed/cleaned_data.csv")

print(f"Dataset shape: {df.shape}")

# Separate features (X) and target (y)
X = df.drop(columns=["churn", "customer_id"], errors="ignore")
y = df["churn"]

X = pd.get_dummies(X, drop_first=True)

print(f"Features shape after encoding: {X.shape}")
print(f"Target class distribution:\n{y.value_counts(normalize=True)}")

# Stratified Train-Test Split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    random_state=42, 
    stratify=y  # Crucial for imbalanced classification
)

print(f"\nTraining set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")
print(f"Train churn rate: {y_train.mean():.4f}")
print(f"Test churn rate: {y_test.mean():.4f}")



Dataset shape: (7043, 24)
Features shape after encoding: (7043, 35)
Target class distribution:
churn
0    0.73463
1    0.26537
Name: proportion, dtype: float64

Training set shape: (5634, 35)
Testing set shape: (1409, 35)
Train churn rate: 0.2654
Test churn rate: 0.2654


In [14]:
# Generate Clusters, Encode, Split, Apply SMOTE, and Save Processed Data
import os
from imblearn.over_sampling import SMOTE
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Create a copy of df features for clustering
X_clustering = df.drop(columns=["churn", "customer_id"], errors="ignore")
X_clustering_encoded = pd.get_dummies(X_clustering, drop_first=True)

# Scale and run K-Means to generate cluster labels
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clustering_encoded)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X_scaled)

# Define features X and target y, and encode ALL categorical columns (including cluster and gender)
X = df.drop(columns=["churn", "customer_id"], errors="ignore")
X = pd.get_dummies(X, columns=["cluster"], drop_first=True)  # ensures cluster dummies
X = pd.get_dummies(X, drop_first=True)  # encodes any remaining text columns
y = df["churn"]

print(f"Updated Features shape including cluster info and encodings: {X.shape}")

# Stratified Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    random_state=42, 
    stratify=y
)

print("\nBefore SMOTE:")
print(f"X_train shape: {X_train.shape}")
print(f"y_train class counts:\n{y_train.value_counts()}")

# Apply SMOTE exclusively to the training set
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("\nAfter SMOTE:")
print(f"X_train_resampled shape: {X_train_resampled.shape}")
print(f"y_train_resampled class counts:\n{y_train_resampled.value_counts()}")

# Save Processed & Resampled Datasets to disk
os.makedirs("../data/processed", exist_ok=True)

train_df = X_train.copy()
train_df["churn"] = y_train.values
train_df.to_csv("../data/processed/train_data.csv", index=False)

test_df = X_test.copy()
test_df["churn"] = y_test.values
test_df.to_csv("../data/processed/test_data.csv", index=False)

resampled_df = X_train_resampled.copy()
resampled_df["churn"] = y_train_resampled.values
resampled_df.to_csv("../data/processed/train_resampled_smote.csv", index=False)

print("\nSuccessfully saved processed files to ../data/processed/:")
print("- train_data.csv")
print("- test_data.csv")
print("- train_resampled_smote.csv")

Updated Features shape including cluster info and encodings: (7043, 38)

Before SMOTE:
X_train shape: (5634, 38)
y_train class counts:
churn
0    4139
1    1495
Name: count, dtype: int64

After SMOTE:
X_train_resampled shape: (8278, 38)
y_train_resampled class counts:
churn
0    4139
1    4139
Name: count, dtype: int64

Successfully saved processed files to ../data/processed/:
- train_data.csv
- test_data.csv
- train_resampled_smote.csv


In [ ]:
# Train and Evaluate 5 Classification Models and Save Results
import os
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "SVM": SVC(probability=True, random_state=42),
    "XGBoost": XGBClassifier(eval_metric="logloss", random_state=42)
}

results = []

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train_resampled, y_train_resampled)
    
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None
    
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob) if y_prob is not None else 0.0
    })

results_df = pd.DataFrame(results).sort_values(by="F1-Score", ascending=False)

os.makedirs("../artifacts/reports", exist_ok=True)
results_df.to_csv("../artifacts/reports/model_comparision.csv", index=False)
print("\nModel comparison successfully saved to ../artifacts/reports/model_comparision.csv")

display(results_df)

Training Logistic Regression...


/home/ememes/ai_projects/client_churn_prediction/venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Training Decision Tree...
Training Random Forest...
Training SVM...


/home/ememes/ai_projects/client_churn_prediction/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Training XGBoost...


,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Logistic Regression,0.777147,0.575000,0.614973,0.594315,0.834930
2,Random Forest,0.773598,0.572178,0.582888,0.577483,0.821861
4,XGBoost,0.767211,0.562500,0.553476,0.557951,0.812910
1,Decision Tree,0.723208,0.482379,0.585561,0.528986,0.681121
3,SVM,0.691980,0.443396,0.628342,0.519912,0.757129


In [16]:
# Section 4 Requirement - Compare Performance With vs. Without Cluster Feature
from sklearn.metrics import f1_score, roc_auc_score

print("--- Evaluating Model Performance: WITHOUT vs. WITH Cluster Feature ---")

# Prepare dataset WITHOUT cluster columns for training & testing
X_train_no_cluster = X_train.loc[:, ~X_train.columns.str.startswith('cluster')]
X_test_no_cluster = X_test.loc[:, ~X_test.columns.str.startswith('cluster')]

# Re-apply SMOTE specifically for the non-cluster training set
X_train_no_cluster_resampled, y_train_no_cluster_resampled = smote.fit_resample(X_train_no_cluster, y_train)

# Train our primary model (e.g., Random Forest or XGBoost) on both versions
from sklearn.ensemble import RandomForestClassifier

# Model WITHOUT cluster feature
rf_no_cluster = RandomForestClassifier(random_state=42)
rf_no_cluster.fit(X_train_no_cluster_resampled, y_train_no_cluster_resampled)
y_pred_no_cluster = rf_no_cluster.predict(X_test_no_cluster)
y_prob_no_cluster = rf_no_cluster.predict_proba(X_test_no_cluster)[:, 1]

f1_no_cluster = f1_score(y_test, y_pred_no_cluster)
roc_no_cluster = roc_auc_score(y_test, y_prob_no_cluster)

# Model WITH cluster feature (from our previous cell, e.g., Random Forest)
rf_with_cluster = RandomForestClassifier(random_state=42)
rf_with_cluster.fit(X_train_resampled, y_train_resampled)
y_pred_with_cluster = rf_with_cluster.predict(X_test)
y_prob_with_cluster = rf_with_cluster.predict_proba(X_test)[:, 1]

f1_with_cluster = f1_score(y_test, y_pred_with_cluster)
roc_with_cluster = roc_auc_score(y_test, y_prob_with_cluster)

# Display comparative metrics
comparison_df = pd.DataFrame([
    {
        "Configuration": "Without Cluster Feature", 
        "F1-Score": f1_no_cluster, 
        "ROC-AUC": roc_no_cluster
    },
    {
        "Configuration": "With Cluster Feature (+Cluster)", 
        "F1-Score": f1_with_cluster, 
        "ROC-AUC": roc_with_cluster
    }
])

display(comparison_df)

--- Evaluating Model Performance: WITHOUT vs. WITH Cluster Feature ---


,Configuration,F1-Score,ROC-AUC
0,Without Cluster Feature,0.570674,0.818274
1,With Cluster Feature (+Cluster),0.577483,0.821861
